# PointillSim: Introduction to the Simulation Framework

This notebook introduces the core concepts and philosophy behind PointillSim, a rule-based simulation engine for imaging-based spatial transcriptomics.

## Contents

1. **Philosophy & Design Principles**
2. **Gene Expression Profiles**: Creating `TissueCellTypes`
3. **The Transfer Function**: Mapping scRNA-seq to spatial data
4. **Field of View (FOV) Generation**: Placing cells in space
5. **Cell Type Probabilities**: Understanding the probability vector field
6. **Realization**: From probabilities to concrete assignments
7. **Cell Morphology**: Physical properties of cells
8. **Transcript Dots**: The observation process
9. **Complete Pipeline**: Putting it all together

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.collections import PatchCollection
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

# Import PointillSim components
from pointillsim import (
    TissueCellTypes,
    CellTypesProperties,
    HybISS_Setup,
    FOV,
    FOVDistribution,
    FrameWideElement,
    HistologicalElement,
    RandomCellTypeRule,
    MixOfNCellTypesRule,
    SingleTypeRule,
    IdentityTransfer,
    AffineNonNegTransfer,
    plot_fov,
    plot_expression_matrix,
)

# Plotting style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'

---
## 1. Philosophy & Design Principles

PointillSim follows three core principles:

### Compositional Architecture
Tissues are built from **histological elements** - discrete regions with defined boundaries. Like building blocks, these elements can be combined to create complex tissue architectures.

### Separation of Concerns
The framework cleanly separates:
- **Where** cells are placed (spatial layout)
- **What** types they become (cell type assignment)
- **How** they are observed (measurement process)

### Probabilistic Ground Truth
Each cell carries a **probability vector** over all cell types, not a hard assignment. This reflects biological reality where:
- Cell identity exists on a continuum
- Transition states and mixed phenotypes are common
- Uncertainty is intrinsic to the system

The **realization** step then samples a concrete cell type from this distribution, mimicking how we observe discrete cell types from a continuous underlying biology.

---
## 2. Gene Expression Profiles

The simulation begins with defining what makes each cell type unique: their **gene expression profiles**.

`TissueCellTypes` stores a matrix of expression levels (genes × cell types). This represents the "reference atlas" - what you might derive from single-cell RNA sequencing.

In [ ]:
# Create a tissue with 3 cell types and 20 genes
# This is a simple example - real tissues would have many more of each

tissue = TissueCellTypes()
tissue.generate_types_and_markers(
    n_genes=20,          # Number of genes in the panel
    n_cell_types=3,      # Number of distinct cell types
    expected_level=8.0,  # Average expression level (counts)
    concentration=0.90,  # Controls sparsity - higher = more marker-like genes
)

print(f"Expression matrix shape: {tissue.gene_expression_by_type.shape}")
print(f"  - {tissue.n_genes} genes")
print(f"  - {tissue.n_cell_types} cell types")

In [ ]:
# Visualize the expression matrix as a heatmap
fig, ax = plot_expression_matrix(
    tissue, 
    figsize=(8, 10),
    log_scale=True,
    title="Gene Expression Reference (log scale)"
)
plt.tight_layout()
plt.show()

In [ ]:
# Let's look at the raw numbers
df = tissue.make_pandas_df()
print("Gene expression matrix (genes x cell types):")
print(df.round(2).to_string())

### Understanding the Expression Matrix

Each column represents a **cell type's expression profile**. Each row represents a **gene**.

Notice how the `generate_types_and_markers()` function creates **marker genes** - genes that are highly expressed in one cell type but low in others. This mimics biological marker genes used for cell type identification.

The `concentration` parameter controls this: higher values create more "pure" markers, lower values create more ubiquitously expressed genes.

In [ ]:
# Compare high vs low concentration
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High concentration (marker-like)
tissue_markers = TissueCellTypes()
tissue_markers.generate_types_and_markers(n_genes=15, n_cell_types=3, concentration=0.99)
ax = axes[0]
im = ax.imshow(tissue_markers.gene_expression_by_type, aspect='auto', cmap='viridis')
ax.set_title('High concentration (0.99)\nStrong marker genes')
ax.set_xlabel('Cell Type')
ax.set_ylabel('Gene')
plt.colorbar(im, ax=ax, label='Expression')

# Low concentration (ubiquitous)
tissue_ubiq = TissueCellTypes()
tissue_ubiq.generate_types_and_markers(n_genes=15, n_cell_types=3, concentration=0.3)
ax = axes[1]
im = ax.imshow(tissue_ubiq.gene_expression_by_type, aspect='auto', cmap='viridis')
ax.set_title('Low concentration (0.3)\nMore ubiquitous expression')
ax.set_xlabel('Cell Type')
ax.set_ylabel('Gene')
plt.colorbar(im, ax=ax, label='Expression')

plt.tight_layout()
plt.show()

---
## 3. The Transfer Function

Real spatial transcriptomics data differs from scRNA-seq references due to:
- **Gene-specific detection efficiency** (some probes work better than others)
- **Systematic biases** (amplification, hybridization efficiency)
- **Non-linear effects** (saturation, background)

The **transfer function** models these effects, transforming the "ideal" expression profile into what we actually observe.

In [ ]:
# Identity transfer - no transformation (ideal case)
identity_tf = IdentityTransfer()

# Affine transfer - models gene-specific scaling and offset
# y = scale * x + offset
affine_tf = AffineNonNegTransfer(
    scales=1.0,       # Mean scale factor
    scales_std=0.3,   # Variation in scale (some genes detected better)
    offsets=0.0,      # Mean offset (background)
    offsets_std=0.1,  # Variation in offset
)

# Apply to expression matrix
raw_expr = tissue.gene_expression_by_type.copy()
transformed_expr = affine_tf.transform(raw_expr)

In [ ]:
# Visualize the effect of the transfer function
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original
ax = axes[0]
im = ax.imshow(np.log1p(raw_expr), aspect='auto', cmap='viridis')
ax.set_title('Original Expression\n(scRNA-seq reference)')
ax.set_xlabel('Cell Type')
ax.set_ylabel('Gene')
plt.colorbar(im, ax=ax)

# Transformed
ax = axes[1]
im = ax.imshow(np.log1p(transformed_expr), aspect='auto', cmap='viridis')
ax.set_title('Transformed Expression\n(after transfer function)')
ax.set_xlabel('Cell Type')
ax.set_ylabel('Gene')
plt.colorbar(im, ax=ax)

# Show the gene-specific scales
ax = axes[2]
ax.barh(range(len(affine_tf.scales)), affine_tf.scales)
ax.set_xlabel('Scale Factor')
ax.set_ylabel('Gene')
ax.set_title('Gene-specific\nDetection Efficiency')
ax.axvline(1.0, color='red', linestyle='--', label='Ideal')
ax.legend()

plt.tight_layout()
plt.show()

---
## 4. Field of View (FOV) Generation

A **Field of View (FOV)** represents a single microscope image - a rectangular region containing cells.

### The FOV Object

An FOV contains:
- `cell_centroids`: (N, 2) array of x, y positions
- `cell_probabilities`: (N, K) array of probabilities over K cell types
- `class_instance_one_hot`: (N, K) one-hot encoding of realized cell types

### FOVDistribution

`FOVDistribution` is a factory that generates FOVs by composing histological elements:
1. A **background element** (fills the entire frame)
2. **Foreground elements** (structures placed on top)

In [ ]:
# Define a simple background using RandomCellTypeRule
# This creates a uniform mixture of our 3 cell types

n_cell_types = 3
frame_size = 500  # 500x500 pixel FOV

# Background element factory (called each time we generate a FOV)
def make_background():
    return FrameWideElement(
        frame_size=frame_size,
        tipical_cell_spacing=15,  # Controls cell density
        rules=RandomCellTypeRule(n_cell_types=n_cell_types)
    )

# Create the FOV distribution
fov_dist = FOVDistribution(
    frame_size=frame_size,
    background_element=make_background,
)

# Generate an FOV
fov = fov_dist.generate_fov()

print(f"Generated FOV with {fov.n_cells} cells")
print(f"Cell centroids shape: {fov.cell_centroids.shape}")
print(f"Cell probabilities shape: {fov.cell_probabilities.shape}")

In [ ]:
# Visualize the FOV - just cell positions for now
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(
    fov.cell_centroids[:, 0], 
    fov.cell_centroids[:, 1],
    s=30, alpha=0.6, c='steelblue'
)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Cell Positions ({fov.n_cells} cells)')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
plt.show()

### Cell Spacing and Density

The `tipical_cell_spacing` parameter controls cell density. Cells are placed on a quasi-hexagonal grid (optimal 2D packing) with jitter added for natural appearance.

In [ ]:
# Compare different cell spacings
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, spacing in zip(axes, [10, 20, 40]):
    fov_temp = FOVDistribution(
        frame_size=frame_size,
        background_element=lambda s=spacing: FrameWideElement(
            frame_size=frame_size,
            tipical_cell_spacing=s,
            rules=RandomCellTypeRule(n_cell_types=3)
        ),
    ).generate_fov()
    
    ax.scatter(fov_temp.cell_centroids[:, 0], fov_temp.cell_centroids[:, 1], s=20, alpha=0.6)
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    ax.set_title(f'Spacing = {spacing}\n({fov_temp.n_cells} cells)')

plt.tight_layout()
plt.show()

---
## 5. Cell Type Probabilities: The Probability Vector Field

This is a key concept in PointillSim: **each cell is not assigned a single cell type, but rather a probability distribution over all cell types**.

Think of the FOV as a "probability field" where each position has an associated probability vector.

In [ ]:
# Look at probabilities for a few cells
print("Cell type probabilities for first 5 cells:")
print("="*50)
for i in range(5):
    probs = fov.cell_probabilities[i]
    print(f"Cell {i}: ", end="")
    for j, p in enumerate(probs):
        print(f"Type{j}={p:.2f}  ", end="")
    print()

In [ ]:
# Visualize the probability field - one plot per cell type
fig, axes = plt.subplots(1, n_cell_types, figsize=(5*n_cell_types, 5))

for i, ax in enumerate(axes):
    scatter = ax.scatter(
        fov.cell_centroids[:, 0],
        fov.cell_centroids[:, 1],
        c=fov.cell_probabilities[:, i],
        cmap='Reds',
        vmin=0, vmax=1,
        s=30, alpha=0.8
    )
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    ax.set_title(f'P(Type {i})')
    plt.colorbar(scatter, ax=ax, label='Probability')

plt.suptitle('Probability Field: Each cell carries a probability vector', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Why Probabilities?

Using probability vectors instead of hard assignments provides:

1. **Biological realism**: Real cells can express markers of multiple types (transitional states)
2. **Uncertainty quantification**: Some cells are "clearly" one type, others are ambiguous
3. **Flexible modeling**: Rules can create gradients and soft boundaries
4. **Ground truth for benchmarking**: Test if methods can recover soft labels

In [ ]:
# Calculate and visualize entropy as a measure of uncertainty
probs = fov.cell_probabilities
entropy = -np.sum(probs * np.log2(probs + 1e-10), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Entropy distribution
ax = axes[0]
ax.hist(entropy, bins=30, edgecolor='black', alpha=0.7)
ax.set_xlabel('Entropy (bits)')
ax.set_ylabel('Number of cells')
ax.set_title('Distribution of Cell Uncertainty')
ax.axvline(np.log2(n_cell_types), color='red', linestyle='--', label=f'Max entropy ({np.log2(n_cell_types):.2f} bits)')
ax.legend()

# Spatial distribution of entropy
ax = axes[1]
scatter = ax.scatter(
    fov.cell_centroids[:, 0],
    fov.cell_centroids[:, 1],
    c=entropy,
    cmap='viridis',
    s=30, alpha=0.8
)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Spatial Distribution of Uncertainty')
plt.colorbar(scatter, ax=ax, label='Entropy (bits)')

plt.tight_layout()
plt.show()

---
## 6. Realization: From Probabilities to Assignments

At some point, we need discrete cell type labels. The **realization** step samples a concrete cell type for each cell from its probability distribution.

This is done using multinomial sampling: each cell independently draws its type.

In [ ]:
# The realization is done automatically when generating FOV
# but we can also call it explicitly:
# fov.realization()

# Get the realized cell types
cell_types = fov.class_instance  # Integer indices
cell_types_onehot = fov.class_instance_one_hot  # One-hot encoding

print("First 10 cells:")
print("Probabilities -> Realized Type")
for i in range(10):
    probs_str = "  ".join([f"{p:.2f}" for p in fov.cell_probabilities[i]])
    print(f"[{probs_str}] -> Type {cell_types[i]}")

In [ ]:
# Compare probability-based coloring vs realized coloring
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Max probability (soft assignment)
ax = axes[0]
max_prob_type = np.argmax(fov.cell_probabilities, axis=1)
scatter = ax.scatter(
    fov.cell_centroids[:, 0],
    fov.cell_centroids[:, 1],
    c=max_prob_type,
    cmap='Set1',
    s=30, alpha=0.8
)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Maximum Probability Assignment\n(deterministic)')
plt.colorbar(scatter, ax=ax, label='Cell Type', ticks=range(n_cell_types))

# Realized (sampled) assignment
ax = axes[1]
scatter = ax.scatter(
    fov.cell_centroids[:, 0],
    fov.cell_centroids[:, 1],
    c=cell_types,
    cmap='Set1',
    s=30, alpha=0.8
)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Realized Assignment\n(sampled from probabilities)')
plt.colorbar(scatter, ax=ax, label='Cell Type', ticks=range(n_cell_types))

plt.tight_layout()
plt.show()

# Count cell types
print("\nCell type counts:")
for t in range(n_cell_types):
    count = np.sum(cell_types == t)
    print(f"  Type {t}: {count} cells ({100*count/len(cell_types):.1f}%)")

---
## 7. Cell Morphology: Physical Properties

Real cells have physical properties beyond position and type:
- **Size** (cell radius)
- **Shape** (anisotropy - elongation)
- **Orientation** (rotation angle)
- **RNA content** (affects number of transcripts detected)

`CellTypesProperties` defines these per cell type, then applies them to individual cells with realistic variation.

In [ ]:
# Create properties for our 3 cell types
# Let's make them different to see the effect

cell_props = CellTypesProperties(
    n_cell_types=n_cell_types,
    sizes=[12, 15, 10],           # Different sizes per type
    size_variation=2,              # Variation within each type
    anisotropy=[0.9, 0.7, 0.85],  # Different elongation (1=round, <1=elongated)
    anisotropy_variation=0.05,
    relative_rna_concentration=[1.0, 1.5, 0.8],  # Different RNA content
    rna_concentration_variation=0.1,
)

# Apply to FOV
cell_props.apply(fov)

# Now the FOV has morphological properties
print("Morphological properties added to FOV:")
print(f"  cell_major_axis: shape {fov.cell_major_axis.shape}")
print(f"  cell_minor_axis: shape {fov.cell_minor_axis.shape}")
print(f"  cell_rotation: shape {fov.cell_rotation.shape}")
print(f"  cell_rna_concentration: shape {fov.cell_rna_concentration.shape}")
print(f"  cell_colors: {len(fov.cell_colors)} colors")

In [ ]:
# Visualize cells as ellipses with their morphological properties

fig, ax = plt.subplots(figsize=(10, 10))

# Create ellipse patches for each cell
ellipses = []
colors = []

for i in range(fov.n_cells):
    ellipse = Ellipse(
        xy=(fov.cell_centroids[i, 0], fov.cell_centroids[i, 1]),
        width=2 * fov.cell_major_axis[i],
        height=2 * fov.cell_minor_axis[i],
        angle=np.degrees(fov.cell_rotation[i]),
    )
    ellipses.append(ellipse)
    colors.append(fov.cell_colors[i])

collection = PatchCollection(ellipses, alpha=0.6)
collection.set_facecolors(colors)
collection.set_edgecolors('black')
collection.set_linewidths(0.5)

ax.add_collection(collection)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Cells with Morphological Properties\n({fov.n_cells} cells)')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=cell_props.colordict[i], alpha=0.6, 
                         label=f'Type {i}') for i in range(n_cell_types)]
ax.legend(handles=legend_elements, loc='upper right')

plt.show()

In [ ]:
# Zoom in on a region to see detail
fig, ax = plt.subplots(figsize=(10, 10))

# Select cells in a 200x200 region
x_min, x_max = 150, 350
y_min, y_max = 150, 350

mask = (
    (fov.cell_centroids[:, 0] >= x_min) & (fov.cell_centroids[:, 0] <= x_max) &
    (fov.cell_centroids[:, 1] >= y_min) & (fov.cell_centroids[:, 1] <= y_max)
)

ellipses = []
colors = []

for i in np.where(mask)[0]:
    ellipse = Ellipse(
        xy=(fov.cell_centroids[i, 0], fov.cell_centroids[i, 1]),
        width=2 * fov.cell_major_axis[i],
        height=2 * fov.cell_minor_axis[i],
        angle=np.degrees(fov.cell_rotation[i]),
    )
    ellipses.append(ellipse)
    colors.append(fov.cell_colors[i])

collection = PatchCollection(ellipses, alpha=0.7)
collection.set_facecolors(colors)
collection.set_edgecolors('black')
collection.set_linewidths(1)

ax.add_collection(collection)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect('equal')
ax.set_title('Zoomed Region - Cell Morphology Detail')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')

plt.show()

---
## 8. Transcript Dots: The Observation Process

In imaging-based spatial transcriptomics, we don't measure continuous expression levels. Instead, we observe **discrete transcript dots** - each representing the detection of a single mRNA molecule.

`HybISS_Setup` models this observation process:
1. Calculate expected transcript counts per cell per gene
2. Apply Poisson sampling (technical noise)
3. Place dots spatially within each cell's boundaries

In [ ]:
# Create HybISS experiment setup
hybiss = HybISS_Setup(
    tissue=tissue,
    genes_sensitivities=1.0,      # Mean detection sensitivity
    genes_sensitivities_variation=0.3,  # Variation (some genes detected better)
    transfer_function=IdentityTransfer(),  # No transformation for simplicity
)

# Generate dots
hybiss.observe_dots(fov)

# Get the results
dots_df = hybiss.make_pandas_df()
cells_df = fov.make_pandas_df()

print(f"Generated {len(dots_df)} transcript dots")
print(f"Across {fov.n_cells} cells")
print(f"Average dots per cell: {len(dots_df)/fov.n_cells:.1f}")

In [ ]:
# Examine the dots DataFrame
print("Dots DataFrame:")
print(dots_df.head(10))
print("\n")
print("Gene counts:")
print(dots_df['gene'].value_counts())

In [ ]:
# Visualize dots colored by gene
fig, ax = plt.subplots(figsize=(12, 10))

# Create gene color mapping
genes = dots_df['gene'].unique()
gene_colors = {gene: plt.cm.tab20(i % 20) for i, gene in enumerate(genes)}

# Plot dots
for gene in genes:
    mask = dots_df['gene'] == gene
    ax.scatter(
        dots_df.loc[mask, 'x'],
        dots_df.loc[mask, 'y'],
        c=[gene_colors[gene]],
        s=3, alpha=0.5,
        label=gene
    )

ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Transcript Dots ({len(dots_df)} total)')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')

# Legend (only first 10 genes to avoid clutter)
ax.legend(loc='upper right', ncol=2, fontsize=8, title='Gene')

plt.show()

In [ ]:
# Show cells with their dots - zoomed region
fig, ax = plt.subplots(figsize=(12, 12))

x_min, x_max = 150, 350
y_min, y_max = 150, 350

# Draw cell ellipses
mask_cells = (
    (fov.cell_centroids[:, 0] >= x_min) & (fov.cell_centroids[:, 0] <= x_max) &
    (fov.cell_centroids[:, 1] >= y_min) & (fov.cell_centroids[:, 1] <= y_max)
)

ellipses = []
colors = []
for i in np.where(mask_cells)[0]:
    ellipse = Ellipse(
        xy=(fov.cell_centroids[i, 0], fov.cell_centroids[i, 1]),
        width=2 * fov.cell_major_axis[i],
        height=2 * fov.cell_minor_axis[i],
        angle=np.degrees(fov.cell_rotation[i]),
    )
    ellipses.append(ellipse)
    colors.append(fov.cell_colors[i])

collection = PatchCollection(ellipses, alpha=0.3)
collection.set_facecolors(colors)
collection.set_edgecolors('black')
collection.set_linewidths(1)
ax.add_collection(collection)

# Draw dots
mask_dots = (
    (dots_df['x'] >= x_min) & (dots_df['x'] <= x_max) &
    (dots_df['y'] >= y_min) & (dots_df['y'] <= y_max)
)

ax.scatter(
    dots_df.loc[mask_dots, 'x'],
    dots_df.loc[mask_dots, 'y'],
    c='black', s=8, alpha=0.7, zorder=5
)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect('equal')
ax.set_title('Cells with Transcript Dots (zoomed)')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')

plt.show()

### Understanding the Observation Model

For each cell, the number of transcripts detected follows:

$$N_{c,g} \sim \text{Poisson}(\lambda_{c,g})$$

where $\lambda_{c,g} = \text{RNA}_c \times E_g \times S_g$

- $\text{RNA}_c$: cell's RNA concentration
- $E_g$: expression level of gene g in cell's type
- $S_g$: gene-specific sensitivity

In [ ]:
# Look at the count matrix
print(f"Cell × Gene count matrix shape: {hybiss.cellxgene_counts.shape}")
print(f"  ({fov.n_cells} cells × {tissue.n_genes} genes)")

# Summary statistics
total_per_cell = hybiss.cellxgene_counts.sum(axis=1)
print(f"\nTranscripts per cell: mean={total_per_cell.mean():.1f}, std={total_per_cell.std():.1f}")
print(f"  min={total_per_cell.min()}, max={total_per_cell.max()}")

---
## 9. Complete Pipeline: Putting It All Together

Let's walk through the complete simulation pipeline from start to finish.

In [ ]:
# =============================================================================
# COMPLETE POINTILLSIM PIPELINE
# =============================================================================

# 1. DEFINE TISSUE EXPRESSION PROFILES
# ------------------------------------
tissue = TissueCellTypes()
tissue.generate_types_and_markers(
    n_genes=30,
    n_cell_types=3,
    expected_level=10.0,
    concentration=0.85,
)
print(f"1. Created tissue with {tissue.n_genes} genes and {tissue.n_cell_types} cell types")

# 2. DEFINE CELL MORPHOLOGICAL PROPERTIES
# ---------------------------------------
cell_props = CellTypesProperties(
    n_cell_types=tissue.n_cell_types,
    sizes=13,
    anisotropy=0.85,
    relative_rna_concentration=1.0,
)
print(f"2. Defined cell morphological properties")

# 3. CREATE FOV DISTRIBUTION
# --------------------------
frame_size = 600
fov_dist = FOVDistribution(
    frame_size=frame_size,
    background_element=lambda: FrameWideElement(
        frame_size=frame_size,
        tipical_cell_spacing=15,
        rules=RandomCellTypeRule(n_cell_types=tissue.n_cell_types)
    ),
)
print(f"3. Created FOV distribution (frame size: {frame_size}x{frame_size})")

# 4. GENERATE FOV
# ---------------
fov = fov_dist.generate_fov()
print(f"4. Generated FOV with {fov.n_cells} cells")

# 5. APPLY MORPHOLOGICAL PROPERTIES
# ----------------------------------
cell_props.apply(fov)
print(f"5. Applied morphological properties")

# 6. SET UP OBSERVATION MODEL
# ---------------------------
hybiss = HybISS_Setup(
    tissue=tissue,
    genes_sensitivities=1.0,
    genes_sensitivities_variation=0.2,
    transfer_function=IdentityTransfer(),
)
print(f"6. Created HybISS observation model")

# 7. GENERATE OBSERVATIONS
# ------------------------
hybiss.observe_dots(fov)
print(f"7. Generated {len(hybiss.make_pandas_df())} transcript dots")

# 8. EXPORT DATA
# --------------
dots_df = hybiss.make_pandas_df()
cells_df = fov.make_pandas_df()
print(f"8. Exported data to DataFrames")
print(f"   - dots_df: {dots_df.shape}")
print(f"   - cells_df: {cells_df.shape}")

In [ ]:
# Final visualization - comprehensive view
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

# A: Expression matrix
ax = axes[0, 0]
im = ax.imshow(np.log1p(tissue.gene_expression_by_type), aspect='auto', cmap='viridis')
ax.set_title('A. Gene Expression Reference')
ax.set_xlabel('Cell Type')
ax.set_ylabel('Gene')
plt.colorbar(im, ax=ax, label='log(Expression+1)')

# B: Cell positions colored by type
ax = axes[0, 1]
scatter = ax.scatter(
    fov.cell_centroids[:, 0],
    fov.cell_centroids[:, 1],
    c=fov.class_instance,
    cmap='Set1',
    s=30, alpha=0.7
)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('B. Cell Positions (by type)')
ax.set_xlabel('X')
ax.set_ylabel('Y')
plt.colorbar(scatter, ax=ax, label='Cell Type')

# C: Cell morphology (zoomed)
ax = axes[1, 0]
x_min, x_max = 200, 400
y_min, y_max = 200, 400
mask = (
    (fov.cell_centroids[:, 0] >= x_min) & (fov.cell_centroids[:, 0] <= x_max) &
    (fov.cell_centroids[:, 1] >= y_min) & (fov.cell_centroids[:, 1] <= y_max)
)
ellipses = []
colors = []
for i in np.where(mask)[0]:
    ellipse = Ellipse(
        xy=(fov.cell_centroids[i, 0], fov.cell_centroids[i, 1]),
        width=2 * fov.cell_major_axis[i],
        height=2 * fov.cell_minor_axis[i],
        angle=np.degrees(fov.cell_rotation[i]),
    )
    ellipses.append(ellipse)
    colors.append(fov.cell_colors[i])
collection = PatchCollection(ellipses, alpha=0.6)
collection.set_facecolors(colors)
collection.set_edgecolors('black')
collection.set_linewidths(0.5)
ax.add_collection(collection)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect('equal')
ax.set_title('C. Cell Morphology (zoomed)')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# D: Transcript dots (zoomed)
ax = axes[1, 1]
# Draw faint cells
collection2 = PatchCollection([e for e in ellipses], alpha=0.2)
collection2.set_facecolors(colors)
collection2.set_edgecolors('gray')
collection2.set_linewidths(0.3)
ax.add_collection(collection2)
# Draw dots
mask_dots = (
    (dots_df['x'] >= x_min) & (dots_df['x'] <= x_max) &
    (dots_df['y'] >= y_min) & (dots_df['y'] <= y_max)
)
ax.scatter(
    dots_df.loc[mask_dots, 'x'],
    dots_df.loc[mask_dots, 'y'],
    c='red', s=5, alpha=0.6
)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect('equal')
ax.set_title(f'D. Transcript Dots (zoomed)')
ax.set_xlabel('X')
ax.set_ylabel('Y')

plt.tight_layout()
plt.show()

---
## Summary

This notebook introduced the core concepts of PointillSim:

| Concept | Class | Purpose |
|---------|-------|---------|
| Expression profiles | `TissueCellTypes` | Define what makes each cell type unique |
| Transfer function | `TransferFunctionBase` | Model detection biases |
| Spatial layout | `FOV`, `FOVDistribution` | Place cells in a field of view |
| Probability field | `cell_probabilities` | Soft cell type assignments |
| Realization | `realization()` | Sample concrete cell types |
| Morphology | `CellTypesProperties` | Physical properties of cells |
| Observation | `HybISS_Setup` | Generate transcript dot coordinates |

### Next Steps

In the next notebook, we'll explore:
- Different histological elements (structures with holes, complex shapes)
- Cell type assignment rules (spatial gradients, distance-based patterns)
- Combining elements to create realistic tissue architectures